## Running VGAE / Graph-PCA & baseline models

- Dim-reduction on features ($x \in \mathbb{R}^{N \times P}$: observation $z \in \mathbb{R}^{N \times D}$: representation, $u \in \mathbb{R}^{N \times 1}$: interpretability as "trajectory / gradient")
- Reconstruction w/ $\sigma(z \cdot z^T)$ for graph reconstruction, regularize $u$ w/ CyCIF graph prior
- Benchmark against K-means clustering & PCA, regular VAE, etc.

In [ ]:
import os
import gc
import sys
import pickle
import gzip

import numpy as np
import cupy as cp
import pandas as pd
import scanpy as sc

import torch
import tifffile
import torch.nn as nn

import networkx as nx
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scipy import sparse
from torch_geometric import utils as pyg_utils

torch.manual_seed(42)

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, gen_graph, configs, dataset, zonation
import baseline, sb_vgae, model_train, model_eval

from torch_geometric.loader import DataLoader

### VGAE (DESI)

[Archived]

####  Result Analysis

[Archived]: snippets for gradient visualization code

#### 2D 

Visualize DESI gradients sorted on a sample posterior `U`:

In [ ]:
def disp_module_spatial(z, height, ncol=4, 
                        panel_size=3, title=None,
                        cmap='turbo'):
    assert z.shape[0] % height == 0,\
         "2D spatial plots should have int height & width"
    n_factors = z.shape[-1]  # dim: [N x factor]
    nrow = n_factors // ncol if n_factors % ncol == 0 else n_factors // ncol + 1
    width = z.shape[0] // height
    
    idx = 0
    fig, axes = plt.subplots(nrow, ncol, figsize=((panel_size+0.2)*ncol, panel_size*nrow), dpi=100)
    for r in range(nrow):
        for c in range(ncol):
            axes[r, c].axis('off')
            if idx < n_factors:
                im = axes[r, c].imshow(z[:, idx].reshape(height, width).T, cmap=cmap)
                axes[r, c].set_title('Factor {}'.format(idx))
                if c == ncol-1:
                    plt.colorbar(im, ax=axes[r,c], fraction=0.03)
            idx += 1
                
    plt.tight_layout()
    plt.suptitle(title, y=1.05, fontsize=20)
    plt.show()


# Visuzlize spatial distribution of Node (n) ->module (z) assignments
# q(b): "discrete" latent factors
disp_module_spatial(latent.qb.detach().cpu().numpy(), height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=4,
                    title=r'$q(b \mid \pi, x, A)$')

# q(z): continuous latent factors
disp_module_spatial(latent.qz_loc.detach().cpu().numpy(), height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=4,
                    title=r'$q(z \mid x, A)$')

# q(z | b): "discrete" latent factors
disp_module_spatial(qz, height=desi_imgs[idx].shape[-1],
                    panel_size=4, ncol=4,
                    title=r'$q(z \mid b, x, A)$')

In [ ]:
# Visualize DESI features sorted by prior temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(u_priors[idx].T.flatten())]

plot.disp_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=10)

plot.disp_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Prior Temperature', nbins=20)

plot.disp_gradients(desi_feature_mat_sorted, 
                        ion_labels, 
                        title='Prior Temperature', nbins=100)


In [ ]:
# Visualize DESI features sorted by POSTERIOR temperature
desi_feature_mat_sorted = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_sorted = desi_feature_mat_sorted[np.argsort(qu.squeeze())] # sort by posterior temp
px_sorted = px[np.argsort(qu.squeeze())]  # sort by posterior temp

plot.disp_gradients(desi_feature_mat_sorted, 
                         ion_labels,
                         title='Posterior Temperature', nbins=10)

plot.disp_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=20)

plot.disp_gradients(desi_feature_mat_sorted, 
                         ion_labels, 
                         title='Posterior Temperature', nbins=100)

In [ ]:
nbins = 100

x_means, x_stds, grad_indices = utils.sort_binned_features(desi_feature_mat_sorted, nbins=nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_img_sorted = desi_imgs[idx][grad_indices, ...]
px_img_sorted = px_chan[grad_indices, ...]

desi_grads_df = pd.DataFrame(x_means,  
                                    index=ion_labels_sorted,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_grads_df.head())
# desi_binned_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients.csv', index=True)
del nbins

In [ ]:
plot.disp_gradients(desi_grads_df.values.T, 
                         ion_labels_sorted, cluster_ions=False,
                         title='Posterior Temperature', nbins=100)

In [ ]:
# Smoothed gradient plots of indiv. features:
from scipy.stats import linregress

nbins = 100
desi_gradient_ts = []  # Fitted slope of the features along PV -> CV trajectory

for x_mean, x_std, label in zip(x_means, x_stds, ion_labels_sorted):
    # plot.disp_gradient(feature_mean, feature_std, title=label)
    ts = linregress(np.linspace(-1, 1, nbins), x_mean)
    desi_gradient_ts.append(ts)

del label, x_mean, x_std

In [ ]:
# Summarize expression gradienst along PV->CV trajectory
desi_gradient_slopes = [ts.slope for ts in desi_gradient_ts]
desi_gradient_pvals = [ts.pvalue for ts in desi_gradient_ts]
desi_gradient_categories = []
for slope, pval in zip(desi_gradient_slopes, desi_gradient_pvals):
    if pval >= 0.05:
        desi_gradient_categories.append('Uncertain')
    elif slope > 0:
        desi_gradient_categories.append('+')
    else:
        desi_gradient_categories.append('-')


In [ ]:
valid_slopes = [slope 
                for (slope, p) in zip(desi_gradient_slopes, desi_gradient_pvals)
                if p < 0.05]

plt.figure(figsize=(5, 2))
plt.hist(valid_slopes, edgecolor='white', bins=10)
plt.xlabel(r'Fitted slope $\beta$')
plt.suptitle('Metabolite gradients along PV->CV trajectory')
plt.show()

del valid_slopes

In [ ]:
# summarize metabolite gradient as slopes
gradient_df = pd.DataFrame({
    'Label': ion_labels_sorted,
    'Slope': desi_gradient_slopes,
    'Category': desi_gradient_categories,
    'p-val': desi_gradient_pvals
})

gradient_df.set_index('Label', inplace=True)
# gradient_df.to_csv('../data/desi/res/metabolite_gradients.csv', index=True)
gradient_df.head()

Check annotated metabolites:

In [ ]:
annot_labels = [
    '865.49527 m/z ± 50 mDa',
    '267.08191 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '261.14629 m/z ± 50 mDa', 
    '253.21007 m/z ± 25.16 mDa', 
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '346.06614 m/z ± 50 mDa',
    '609.52634 m/z ± 50 mDa',
    '699.51653 m/z ± 50 mDa'
]


In [ ]:
annot_indices = []
for i, (x_mean, x_std, label) in enumerate(zip(x_means, 
                                               x_stds, 
                                               ion_labels_sorted)):
    if label in annot_labaels:
        annot_indices.append(i)

        # Trajectory plot
        plot.disp_gradient(x_mean, x_std, title=label)

        # Feature image plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=100)
        ax1.imshow(desi_img_sorted[i], cmap='magma')
        ax1.set_title(label + '\n(raw)')
        ax2.imshow(px_img_sorted[i], cmap='magma')
        ax2.set_title(label + '\n(reconst)')
        plt.tight_layout()
        plt.show()

del label, x_mean, x_std

In [ ]:
# Try posterior denoised expression p(x|z)
nbins = 100
px_means, px_stds, grad_indices = utils.sort_binned_features(px_sorted, nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_img_sorted = desi_imgs[idx][grad_indices, ...]
px_img_sorted = px_chan[grad_indices, ...]

desi_grads_df = pd.DataFrame(px_means,  
                             index=ion_labels_sorted,
                             columns=['bin' + str(i) for i in range(nbins)])


del nbins

In [ ]:
annot_indices = []
for i, (px_mean, px_std, label) in enumerate(zip(px_means, 
                                               px_stds, 
                                               ion_labels_sorted)):
    if label in annot_labels:
        annot_indices.append(i)
        # Trajectory plot
        plot.disp_gradient(px_mean, px_std, vmax=0.4, title=label)

        # feature image
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6), dpi=100)
        ax1.imshow(desi_img_sorted[i], cmap='magma')
        ax1.set_title(label + '\n(raw)')
        ax2.imshow(px_img_sorted[i], cmap='magma')
        ax2.set_title(label + '\n(reconst)')
        plt.tight_layout()
        plt.show()

del label, px_mean, px_std

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_grads_df.loc[annot_labels].T.corr(), 
                   method='ward',
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()


g = sns.clustermap(desi_grads_df.T.corr(), 
                   method='ward',
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n all DESI features', y=1.3, fontsize=20)
plt.show()

In [ ]:
# Raw-expresssion correlations 
g = sns.clustermap(np.corrcoef(desi_feature_mat_sorted[:, annot_indices].T),
                   method='ward',
                   xticklabels=ion_labels_sorted[annot_indices],
                   yticklabels=ion_labels_sorted[annot_indices],
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(np.corrcoef(desi_feature_mat_sorted.T),
                   method='ward',
                   #xticklabels=ion_labels_sorted,
                   #yticklabels=ion_labels_sorted)
                   cmap='coolwarm'
                  )
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n all DESI features', y=1.3, fontsize=20)
plt.show()

#### 3D integration

- Cluster metabolites with similar trajectory
- Check 3D patterning for annotated metabolites (whether necessary to perform any smoothing to account for noises)

Infer latents across slices:

In [ ]:
# Load data:
# ...

In [ ]:
# Load model:
# ...

In [ ]:
# # Common graph structure for each L-slice
G_desi = gen_graph.construct_graph(desi_coords, k=4)  

b_list = []  # Binary spatial cluster assignment
z_list = []  # Spatial node cluster strength assignment
u_list = []  # Trajectory assignment 
x_3d = []
px_3d = []
module_weights = model.encoder.x_to_zloc.weight.detach().cpu().numpy()

with torch.no_grad():
    for i, desi_img in enumerate(desi_imgs):
        x = desi_img.transpose(2, 1, 0).reshape(-1, nchans)
        x = utils.norm_features(x)
        x_3d.append(x)
        
        latent, recon = model_eval.eval(model, G_desi, x)
        b = latent.qb.detach().cpu().numpy()
        z = latent.qz.detach().cpu().numpy()
        u = latent.qu.detach().cpu().numpy()
        w = model.decoder.z_to_xloc[0].weight.detach().cpu().numpy()
        px = recon.px_loc.detach().cpu().numpy()
        
        b_list.append(b)
        z_list.append(z)
        u_list.append(u)
        module_weights_list.append(w)
        px_3d.append(px)
        
del desi_img, latent, recon, b, z, u, w, x

In [ ]:
for qz in z_list:
    # q(z | b): "discrete" latent factors
    disp_module_spatial(qz, height=ndimy,
                        panel_size=4, ncol=4,
                        title=r'$q(z \mid b, x, A)$')
del qz

In [ ]:
u_post_list_2d = [u.reshape(ndimy, ndimx).T for u in u_list]
plot.disp_chans(u_post_list_2d, cmap='coolwarm', title=r'Zonation posterior ($u$)')

u_post_discrete_2d = [utils.infer_zones(u_post, nbins=10) for u_post in u_post_list_2d]
plot.disp_chans(u_post_discrete_2d, cmap='coolwarm', title=r'Discretized zonation posterior ($u$)')

tifffile.imwrite('../data/desi/res/u_posteriors.tif', np.array(u_post_list_2d))
tifffile.imwrite('../data/desi/res/u_posteriors_discretized.tif', np.array(u_post_discrete_2d))

Check 3D trajectory of representative metabolites:

In [ ]:
x_3d = np.array(x_3d)  # dim: [L, Y*X, C]
x_3d_unsorted = x_3d.transpose(2, 0, 1).reshape(nchans, -1)  # dim: [C, L*Y*X] unsorted
x_3d_sorted = np.zeros_like(x_3d_unsorted)  # expressions sorted by inferred gradient values, dim: [C, L*Y*X]

indices = np.argsort(np.array(u_list).squeeze().flatten())
for i, chan in enumerate(x_3d_unsorted):
    x_3d_sorted[i] = chan[indices]

del x_3d_unsorted, indices, chan

In [ ]:
# Visualize binned trajectory:
plot.disp_gradients(x_3d_sorted.T, 
                         ion_labels, 
                         title='Posterior gradient (3D)', nbins=100)

In [ ]:
# Full annotation labels - separation btw/ categories

FA_labels = [
    '193.11679 m/z ± 50 mDa',
    '195.14353 m/z ± 22.09 mDa',
    '253.21007 m/z ± 25.16 mDa',
    '255.22685 m/z ± 50 mDa',
    '277.22812 m/z ± 39.49 mDa',
    '279.2325 m/z ± 50 mDa',
    '281.2441 m/z ± 39.78 mDa',
    '283.26292 m/z ± 50 mDa',
    '287.21503 m/z ± 50 mDa',
    '289.17446 m/z ± 50 mDa',
    '291.19452 m/z ± 50 mDa',
    '293.2216 m/z ± 50 mDa',
    '295.22855 m/z ± 50 mDa',
    '297.05155 m/z ± 50 mDa',
    '301.22061 m/z ± 50 mDa',
    '303.24218 m/z ± 50 mDa',
    '305.25545 m/z ± 50 mDa',
    '307.27538 m/z ± 50 mDa',
    '311.22364 m/z ± 41.84 mDa',
    '311.30732 m/z ± 41.84 mDa',
    '313.2352 m/z ± 50 mDa',
    '315.1971 m/z ± 28.07 mDa',
    '315.25325 m/z ± 28.07 mDa',
    '317.22145 m/z ± 50 mDa',
    '319.22402 m/z ± 50 mDa',
    '325.21252 m/z ± 50 mDa',
    '327.24012 m/z ± 50 mDa',
    '329.24533 m/z ± 50 mDa',
    '331.25667 m/z ± 50 mDa',
    '333.2164 m/z ± 50 mDa',
    '333.27413 m/z ± 50 mDa',
    '335.23981 m/z ± 50 mDa',
    '339.21764 m/z ± 43.68 mDa',
    '345.24306 m/z ± 50 mDa',
    '349.22053 m/z ± 50 mDa',
    '351.23262 m/z ± 50 mDa',
]

ST_labels = [
    '309.21855 m/z ± 27.8 mDa',
    '343.21893 m/z ± 50 mDa',
    '355.27415 m/z ± 50 mDa',
    '417.34599 m/z ± 50 mDa',
    '448.32793 m/z ± 50 mDa',
    '477.37418 m/z ± 50 mDa',
    '493.32723 m/z ± 50 mDa',
    '498.29133 m/z ± 50 mDa',
]

PE_labels = [
    '714.52524 m/z ± 50 mDa',  # LPE! Lyso-PE (1 FA)
    '738.51815 m/z ± 50 mDa',
    '740.53903 m/z ± 50 mDa',
    '742.51959 m/z ± 50 mDa',
    '762.51427 m/z ± 50 mDa',
    '764.52398 m/z ± 50 mDa',
    '766.53633 m/z ± 50 mDa',
]

PA_labels = [
    '723.47084 m/z ± 50 mDa',
    '817.52389 m/z ± 50 mDa',
    '819.51426 m/z ± 50 mDa'
]

PI_labels = [
    '861.54597 m/z ± 50 mDa',
    '863.58919 m/z ± 50 mDa',
    '885.56543 m/z ± 50 mDa'
]


In [ ]:
# Compute binned trajectory
nbins = 100
x_means, x_stds, grad_indices = utils.sort_binned_features(x_3d_sorted.T, nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_imgs_sorted = [img[grad_indices, ...] for img in desi_imgs]

desi_grads_df = pd.DataFrame(x_means,  
                             index=ion_labels_sorted,
                             columns=['bin' + str(i) for i in range(nbins)])

# display(desi_grads_df.head())
# desi_grads_df.to_csv('../data/desi/res/metabolite_binned_gradients_3d.csv', index=True)
del nbins

In [ ]:
for i, (x_mean, x_std, label) in enumerate(zip(x_means, x_stds, ion_labels_sorted)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_gradient(x_mean, x_std, title=label + ' (3D)')

        plt.figure(figsize=(6, 6), dpi=80)
        plt.imshow(desi_imgs_sorted[10][i], cmap='magma')
        plt.title(label)
        plt.axis('off')
        plt.show()
        
del x_mean, x_std, label

In [ ]:
# Module weight correlation
module_weights = model.encoder.x_to_zloc.weight.detach().cpu().numpy()
module_weights_df = pd.DataFrame(module_weights,
                                 index=ion_labels,
                                 columns=['Factor_{0}'.format(i) for i in range(module_weights.shape[1])])

In [ ]:
g = sns.clustermap(module_weights_df, method='ward', cmap='coolwarm', col_cluster=False)
g.ax_heatmap.axes.set_title('Metabolite module contribution', y=1.3, fontsize=20)
plt.show()

# Correlation of annotated labels
# Similarly, try other module label groups
g = sns.clustermap(module_weights_df.loc[annot_labels].T.corr(),
                   method='ward',
                   #vmin=-1, 
                   #vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of Module weights\n Highlighted features', y=1.3, fontsize=20)
plt.show()


In [ ]:
g = sns.clustermap(module_weights_df.loc[FA_labels + ST_labels + PA_labels + PE_labels + PI_labels].T.cov(),
                   method='ward',
                   # vmin=-1,
                   # vmax=1,
                   figsize=(15, 15),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Covariance of module assignments\n Annotated features', y=1.3, fontsize=15)

g = sns.clustermap(module_weights_df.T.cov(),
                   method='ward',
                   # vmin=-1,
                   # vmax=1,
                   figsize=(15, 15),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Covariance of module assignments\n All features', y=1.3, fontsize=30)

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_grads_df.loc[FA_labels].T.corr(),
                   #vmin=-1, 
                   #vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (FA)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.loc[ST_labels].T.corr(),
                   #vmin=-1, 
                   #vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (ST)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.loc[PE_labels].T.corr(),
                   #vmin=-1, 
                   #vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n (PE)', y=1.3, fontsize=20)
plt.show()

g = sns.clustermap(desi_grads_df.T.corr(),
                   #vmin=-1, 
                   #vmax=1,
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions', y=1.3, fontsize=20)
plt.show()

In [ ]:
x_3d_df = pd.DataFrame(x_3d_sorted, index=ion_labels)

In [ ]:
# # Raw-expresssion correlations 
g = sns.clustermap(np.corrcoef(x_3d_sorted[annot_indices, :]),
                    xticklabels=ion_labels_sorted[annot_indices],
                    yticklabels=ion_labels_sorted[annot_indices],
                    cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of raw-expressions\n btw annotated DESI features', y=1.3, fontsize=20)
plt.show()

Group modules by (1). Module weight assignment; (2).binned trajectories via hierarchical clustering:

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

In [ ]:
# Cluster by Module weight assignment `W_xz`
linked_W_xz = linkage(module_weights_df, method='ward')

fig, ax = plt.subplots(figsize=(8, 20), dpi=500)
dendrogram(linked_W_xz, labels=ion_labels, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
fig.savefig('../data/desi/res/metabolite_module_W_xz.pdf', bbox_inches='tight', dpi=300)

In [ ]:
linked_grads = linkage(desi_grads_df, method='ward')

fig, ax = plt.subplots(figsize=(6, 18), dpi=300)
dendrogram(linked_grads, labels=desi_grads_df.index, color_threshold=3, 
           orientation='left', distance_sort='descending',
           show_leaf_counts=True, ax=ax)
plt.show()
fig.savefig('../data/desi/res/metabolite_module_gradients.pdf', bbox_inches='tight', dpi=300)

In [ ]:
# Tree-cut for clusters
max_d = 3
clusters = fcluster(linked_grads, max_d, criterion='distance')
desi_grads_df.insert(0, 'Cluster', clusters)
display(desi_grads_df.head())
desi_grads_df.to_csv('../data/desi/res/mz_modules_3d.csv', index=True)

Is there any **3D gradient effects** from metabolite?

#### Benchmarks
Baseline methods on full 3D data

##### GASTON

In [ ]:
import gaston
from gaston import process_NN_output, dp_related, cluster_plotting

In [ ]:
gaston_model, A, S = process_NN_output.process_files('../data/desi/res/GASTON_output/')
# S = np.flip(S, axis=1)

In [ ]:
n_layers = 9
u_gaston, gaston_labels = dp_related.get_isodepth_labels(gaston_model, A, S, n_layers)
u_gaston = u_gaston.reshape(-1, desi_img.shape[-1])  # Convert to dim=[i, j]
u_gaston = -u_gaston  # tmp: Convert to relatively correct PV->CV trajectory

In [ ]:
plt.figure()
plt.imshow(u_gaston, cmap='turbo')
plt.colorbar()
plt.axis('off')
plt.title('GASTON isodepth', fontsize=15)
plt.show()

In [ ]:
# Visualize DESI features sorted by GASTON trajectory
desi_feature_mat_gaston = desi_feature_mat.reshape(-1, desi_feature_mat.shape[-1])  # Coord x Feature
desi_feature_mat_gaston = desi_feature_mat_gaston[np.argsort(u_gaston.T.flatten())]

plot.disp_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=10)

plot.disp_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=20)

plot.disp_gradients(desi_feature_mat_gaston, 
                         ion_labels, 
                         title='GASTON isodepth', nbins=100)


In [ ]:
# Plot example metabolites along GASTON isodepth

nbins = 100
# Feature x nbins
desi_gaston_grads, desi_gaston_stds = utils.get_binned_features(desi_feature_mat_gaston, nbins=nbins) 
desi_gaston_grads, desi_gaston_stds = desi_gaston_grads.T, desi_gaston_stds.T

# Sort metabolites based on argmax values along the PV->CV binned trajectory
argmax_grad = desi_gaston_grads.argmax(1)

gaston_ion_labels_sorted = np.array(ion_labels)[np.argsort(argmax_grad)]
gaston_desi_img_sorted = desi_img[np.argsort(argmax_grad), ...]
desi_gaston_grads = desi_gaston_grads[np.argsort(argmax_grad), :]
desi_gaston_stds = desi_gaston_stds[np.argsort(argmax_grad), :]

desi_gaston_grads_df = pd.DataFrame(desi_gaston_grads,  
                                    index=gaston_ion_labels_sorted,
                                    columns=['bin' + str(i) for i in range(nbins)])

display(desi_gaston_grads_df.head())
del nbins

In [ ]:
for i, (feature_mean, feature_std, label) in enumerate(zip(desi_gaston_grads, desi_gaston_stds, gaston_ion_labels_sorted)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_gradient(feature_mean, feature_std, title=label)

        # DESI feature image
        plt.figure(figsize=(6, 6), dpi=100)
        plt.imshow(gaston_desi_img_sorted[i], cmap='magma')
        plt.title(label)
        plt.show()

        
del label, feature_mean, feature_std
del desi_gaston_grads, desi_gaston_stds, gaston_ion_labels_sorted

##### PCA

In [ ]:
# Check raw-example & PCA trajectory covariances

In [ ]:
idx = 10
nchans, ndimy, ndimx = desi_imgs[idx].shape
desi_feature = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)  # dim: [YxX, C]
cov_df = pd.DataFrame(np.cov(desi_feature.T), index=ion_labels, columns=ion_labels)
cov_df.head()

In [ ]:
sns.clustermap(cov_df, method='ward')

In [ ]:
adata = sc.AnnData(desi_feature)
sc.pp.scale(adata)
sc.pp.pca(adata, n_comps=10)
sc.pp.neighbors(adata, n_neighbors=24)  # 2-"hop"
sc.tl.diffmap(adata, n_comps=10)
u_pc1 = adata.obsm['X_pca'][:, 0]
u_dc1 = adata.obsm['X_diffmap'][:, 0]

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
ax1.imshow(u_pc1.reshape(ndimy, ndimx).T, cmap='turbo')
ax1.axis('off')
ax1.set_title('PC1')

ax2.imshow(u_dc1.reshape(ndimy, ndimx).T, cmap='turbo')
ax2.axis('off')
ax2.set_title('DC1')

plt.tight_layout()
plt.show()

In [ ]:
x_3d_unsorted = x_3d.transpose(2, 0, 1).reshape(nchans, -1)  # dim: [C, L*Y*X] unsorted
x_3d_sorted = np.zeros_like(x_3d_unsorted)  # expressions sorted by inferred gradient values, dim: [C, L*Y*X]

indices = np.argsort(np.array(u_pc1s).squeeze().flatten())
for i, chan in enumerate(x_3d_unsorted):
    x_3d_sorted[i] = chan[indices]

del x_3d_unsorted, indices, chan

In [ ]:
# Visualize DESI trajectory represented as PC0
plot.disp_gradients(x_3d_sorted.T, 
                         ion_labels, 
                         title='PC1 isodepth', nbins=100)


In [ ]:
# Compute binned trajectory
nbins = 100
x_pc_means, x_pc_stds, grad_indices = utils.sort_binned_features(x_3d_sorted.T, nbins)

ion_labels_sorted = np.array(ion_labels)[grad_indices]
desi_imgs_sorted = [img[grad_indices, ...] for img in desi_imgs]

desi_pc_grads_df = pd.DataFrame(x_pc_means,  
                             index=ion_labels_sorted,
                             columns=['bin' + str(i) for i in range(nbins)])
del nbins

In [ ]:
for i, (x_pc_mean, x_pc_std, label) in enumerate(zip(x_pc_means, x_pc_stds, ion_labels_sorted)):
    if label in annot_labels:
        # Trajectory plot
        plot.disp_gradient(x_pc_mean, x_pc_std, title=label + ' (PC1 3D)')
        
del label, x_pc_mean, x_pc_std

In [ ]:
# Binned-expression correlations
g = sns.clustermap(desi_pc_grads_df.loc[annot_labels].T.corr(),
                   cmap='coolwarm', square=True)
g.ax_heatmap.axes.set_title('Correlations of binned-expressions\n Annotated DESI features', y=1.3, fontsize=20)
plt.show()


##### Clustering
e.g. K-means / Louvain / Leiden

In [ ]:
idx = 10
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
adata_x = sc.AnnData(desi_features)

sc.pp.pca(adata_x)
sc.pp.neighbors(adata_x, n_neighbors=40)
sc.tl.louvain(adata_x)
adata_x.obs.louvain = adata_x.obs.louvain.astype(np.uint8)

In [ ]:
# Use cluster assignment to represent spatial node modules
u_louvain = adata_x.obs.louvain.to_numpy().reshape(ndimy, ndimx).T
u_louvain_one_hot = [(u_louvain == i).astype(np.uint8)
                     for i in np.sort(adata_x.obs.louvain.unique())]  

plot.disp_chans(u_louvain_one_hot, cmap='magma')
del adata_x

In [ ]:
from sklearn.cluster import KMeans

K = 8
kmeans = KMeans(n_clusters=K, random_state=0, n_init="auto").fit(desi_features)
kmeans_2d = kmeans.labels_.reshape(ndimy, ndimx).T
kmeans_discrete = [kmeans_2d == i for i in range(K)]
del K

plot.disp_chans(kmeans_discrete, title='K-means', ncols=4)

##### LDA

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation as LDA

In [ ]:
idx = 8
desi_features = desi_imgs[idx].transpose(2, 1, 0).reshape(-1, nchans)
desi_features[desi_features < 0] = 0
lda = LDA(n_components=4, random_state=42)
lda.fit(desi_features)

In [ ]:
theta = lda.transform(desi_features)
beta = lda.components_.T

In [ ]:
for i in range(lda.n_components):
    plt.figure()
    plt.imshow(theta[:, i].reshape(ndimy, ndimx).T, cmap='magma')
    plt.show()

### VGAE (Xenium-DESI integration)

- Integrate DESI prior to guide latent prob. clustering of Xenium

In [ ]:
import json
import squidpy as sq

from util import IO, utils, gen_graph
from models import baseline, dataset
from torch_geometric import utils as pyg_utils

from importlib import reload
reload(IO)
reload(utils)
reload(baseline)
reload(gen_graph)
reload(dataset)

torch.manual_seed(0)

In [ ]:
# Load paired Xenium & DESI
# Sample ID: NIH_F5

xenium_path = '../data/xenium/'
desi_path = '../data/integrated/desi_xenium_map/'
sample_id = 'NIH_F5'
# sample_ids = sorted([f for f in os.listdir(xenium_path) if os.path.isdir(os.path.join(xenium_path, f))])

# Load xenium
adata = IO.load_xenium(os.path.join(xenium_path, sample_id))
with open(os.path.join('../data/xenium/', sample_id, 'experiment.xenium'), 'r') as ifile:
    scalefactor = json.load(ifile)['pixel_size']
xenium_coords = adata.obs[['y_centroid', 'x_centroid']].copy().to_numpy()  # YX-index


In [ ]:
# Load DESI
adata_desi_raw, desi_img = IO.load_desi(os.path.join(desi_path, sample_id+'.ome.tif'),
                                        dilate=True)
mz_labels = adata_desi_raw.var_names
del adata_desi_raw

# Filter DESI pixels mapped to Xenium
ratio = 1/10  # xenium --> DESI downscale ratio
desi_coords = np.round(xenium_coords / scalefactor * ratio).astype(np.int16)  # YX-index
desi_features = np.zeros((desi_coords.shape[0], desi_img.shape[0]), dtype=np.float32)
for i, chan in enumerate(desi_img):
    desi_features[:, i] = chan[tuple(desi_coords.T)]

adata_desi = sc.AnnData(desi_features)
adata_desi.obs['x_centroid'], adata_desi.obs['y_centroid'] = desi_coords[:, 1], desi_coords[:, 0]
adata_desi.obsm['spatial'] = adata_desi.obs[['x_centroid', 'y_centroid']].values
adata_desi.var_names = mz_labels
IO.load_spatial(adata_desi, load_img=False)  # Add dummy `adata.uns.spatial

del chan, desi_features

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

def get_PCs(adata, coords, 
            n_pcs, k=24, 
            graph_regularize=False):
    """
    Dimension reduction w/ (graph-regularized) PCA
    """
    if graph_regularize:
        G = gen_graph.construct_graph(coords, k=k, weighted=False)  
        edge_index = pyg_utils.from_networkx(G).edge_index
        model = baseline.GPCALayer(c_in=adata.X.shape[-1],
                                   c_out=n_pcs,
                                   center=True,
                                   init_weight=True,
                                   ortho_weight=True)
        U_gpca = model(torch.tensor(adata.X).float(), edge_index)
        adata.obsm['X_pca'] = U_gpca.detach().cpu().numpy()
    else:
        sc.pp.pca(adata, n_pcs)
    return None
    

In [ ]:
# Compute aux. DESI expressions `u`
# Experiment: whether Encode low-dim representations (e.g. PCA) or raw DESI expressions

# n_aux = 30
# get_PCs(adata_desi, desi_coords, 
#         n_pcs=n_aux,
#         graph_regularize=True)

# sc.pp.neighbors(adata_desi, n_neighbors=30, use_rep='X_pca')
# sc.tl.leiden(adata_desi, resolution=0.1)
# print(adata_desi.obs.leiden.value_counts())
# sc.pl.spatial(adata_desi, color='leiden', size=5)

# Append aux. DESI expression (PC) to Xenium
# adata.obsm['X_aux'] = adata_desi.obsm['X_pca']  
adata.obsm['X_aux'] = adata_desi.X

#### Model sketch iVAE: 
- $z_i \mid u_i \sim \mathcal{LN}(f_{\lambda}(u_i), \sigma^2I)$
- $x_i \mid z_i, \mathbf{A} \sim \mathcal{NegBinom}(l \cdot f_{\theta}(z_i, \mathbf{A}, \theta_g)$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import pyro
import pyro.poutine as poutine
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, TraceEnum_ELBO
from pyro.optim import Adam

from torch_geometric.data import DataLoader

from models import logit_vgae, model_train

In [ ]:
xenium_loader = dataset.XeniumGraphDataset(k=30, n_subgraphs=8) 
xenium_data = xenium_loader.load_graphs([adata])
xenium_train_dl = DataLoader(xenium_data, shuffle=True)

In [ ]:
from importlib import reload

reload(IO)
reload(utils)
reload(plot)
reload(dataset)
reload(configs)
reload(logit_vgae)
reload(model_train)

In [ ]:
lr = 1e-3
n_aux = adata.obsm['X_aux'].shape[1]
n_epochs = 1000
device = torch.device('cuda')

train_configs = configs.set_train_configs(n_epochs=n_epochs, 
                                          lr=lr,
                                          device=device)

model_configs = configs.set_model_configs(device=device,
                                          beta=0.5,
                                          c_in=adata.X.A.shape[-1],
                                          c_aux=n_aux,
                                          c_hidden=32,
                                          c_latent=7,
                                          k_hop=3,
                                          dropout=0.5) 

In [ ]:
pyro.clear_param_store()
model = logit_vgae.LogitVGAE(model_configs, device=train_configs.device)
model, losses = model_train.train_logit_vgae(model, xenium_train_dl, train_configs)

In [ ]:
plt.figure(figsize=(5, 2))
plt.plot(np.arange(train_configs.n_epochs), losses)
plt.xlabel('Epochs')
plt.ylabel('Train ELBO')
plt.show()

In [ ]:
# Inference on full data (CPU)

G_xenium = gen_graph.construct_graph(xenium_coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G_xenium).edge_index

model.device = 'cpu'
model = model.to('cpu')

qz, qz_mu = model.get_z(torch.tensor(adata.X.A).to('cpu'), 
                        torch.tensor(adata.obsm['X_aux']).to('cpu'), 
                        edge_index.to('cpu'))

pz = model.pz_u(torch.tensor(adata.obsm['X_aux']).to('cpu'),
                edge_index.to('cpu'))
px = model.get_x(torch.tensor(adata.X.A).to('cpu'), 
                 edge_index.to('cpu'), 
                 qz_mu)

qz_mu = qz_mu.detach().cpu().numpy()
qz = qz.detach().cpu().numpy()

pz = pz.detach().cpu().numpy()
px = px.detach().cpu().numpy()

del G_xenium
gc.collect()
torch.cuda.empty_cache()

In [ ]:
rand_indices = np.random.choice(np.arange(adata.shape[1]), size=200, replace=False)

plt.figure(figsize=(6, 4))
plt.scatter(adata.X.A[:, rand_indices].flatten(), px[:, rand_indices].flatten(), s=1)
plt.xlabel('X')
plt.ylabel("X_reconst")
plt.show()

del rand_indices

In [ ]:
# Check posterior collapse?
plot.disp_spatial_latents(adata, pz, ncols=4)

In [ ]:
plot.disp_spatial_latents(adata, qz, ncols=4)

**DEBUG**: <br>

$p(z \mid u)$ stable & disentangled, but $q(z \mid x, u)$ persists w/ identical latent factors
- Not fitting well (KL term?)

#### Trajectory Inference

- Try both Xenium & mapped DESI

In [ ]:
adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm)
sc.pp.log1p(adata_norm)
# sc.pp.pca(adata_norm)

adata_qz = sc.AnnData(qz)
sc.pp.pca(adata_qz)
sc.pp.neighbors(adata_qz, n_neighbors=30)
sc.tl.umap(adata_qz)
sc.tl.tsne(adata_qz)

# # Assign learnt embeddings on latent to Xenium / DESI
adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_norm.obsm['X_tsne'] = adata_qz.obsm['X_tsne']
adata_norm.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_norm.obsm['X_z'] = qz

adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']
adata_desi.obsm['X_tsne'] = adata_qz.obsm['X_tsne']
adata_desi.obsm['X_umap'] = adata_qz.obsm['X_umap']
adata_desi.obsm['X_z'] = qz


In [ ]:
# PC on raw Xenium expressions

# sc.pp.pca(adata_norm)
# sc.pl.pca(adata_norm,
#           color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
#                  'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
#                  'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
#                 ],
#           size=10, 
#           cmap='RdBu_r')


In [ ]:
# Separable Zs on TSNE?
z_labels = []
for i in range(qz.shape[1]):
    label = 'z'+str(i)
    adata_norm.obs[label] = qz[:, i]
    z_labels.append(label)
del label

In [ ]:
sc.pl.pca(adata_norm, color=z_labels, cmap='seismic')

In [ ]:
# Xenium markers
sc.pl.pca(adata_norm,
          color=['ADH1C', 'CYP3A4', 'APOA5', 'AQP9',  # PC markers
                 'CYP2A7', 'CYP2B6', 'HPX', 'HMGCS2',  # PP markers
                 'DPT', 'PTGDS', 'MYH11', 'CD34' # Portal-related
                ],
          size=10, 
          cmap='RdBu_r')


In [ ]:
# DESI markers
sc.pl.pca(adata_desi,
          color=['FA 20:4;O',
                 'FA 18:1; O',
                 'FA 22:5;O',
                 '808.6024 m/z ± 30 ppm',
                 '865.50838 m/z ± 50 ppm',
                 '673.49987 m/z ± 30 ppm',
                 '631.4937 m/z ± 30 ppm',
                 '346.05909 m/z ± 40 ppm',
                 '754.5545 m/z ± 30 ppm',
                 '258.11631 m/z ± 30 ppm',
                 '734.58637 m/z ± 30 ppm',
                 '703.58429 m/z ± 30 ppm',
                ],
          size=10, 
          cmap='RdBu_r')

In [ ]:
adata_qz.write_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

**Wasserstein Distance**

First define a global OT distance btw clusters:

In [ ]:
import networkx
from scipy.stats import wasserstein_distance


In [ ]:
# Compute cluster-level "trajectory" via DFS on NNs?
def dfs(mat, start, order='descending', visited=None):
    """
    Compute cluster-level "trajectory" via DFS 
    on nearest neighbor clusters
    """
    assert order == 'ascending' or order == 'descending'
    if visited is None:
        visited = []
    visited.append(start)
    
    # Visit all the neighbors
    neighbors = np.argsort(mat[start]) if order == 'ascending' else np.argsort(mat[start])[::-1]
    neighbors = np.delete(neighbors, np.argwhere(neighbors == start))  # avoid self
    for neighbor in neighbors:
        if neighbor not in visited:
            dfs(mat, neighbor, order, visited)
    
    return visited


def get_wass_dist(m, n):
    x = m / m.max()
    y = n / n.max()
    return wasserstein_distance(x, y)


def get_latent_dists(latent):
    """Relative `distance` btw clusters"""
    ndim = latent.shape[1]
    mat = np.zeros((ndim, ndim), dtype=np.float32)
    for i in range(ndim-1):
        for j in range(i+1, ndim):
            mat[i, j] = get_wass_dist(latent[:, i], latent[:, j])
    return mat+mat.T  # Complete Upp-tril to full symmetric matrix.


def sort_latent(adata, root_marker):
    """
    Sort soft cluster assignments (z) by 
    Wasserstein distance to the root marker 
    """
    assert root_marker in adata.var_names, \
        "Root marker {} doesn't exist".format(root_marker)
    assert 'X_z' in adata.obsm, \
        "Please run the model first the obtain the latent representation"

    z = adata.obsm['X_z']
    dists_to_root = []
    root_expr = adata[:, root_marker].X
    if not isinstance(root_expr, np.ndarray):
        root_expr = root_expr.A
    for zi in z.T:
        dists_to_root.append(get_wass_dist(root_expr.squeeze(), zi))

    return z[:, np.argsort(dists_to_root)]

Need to define at least starting point w/ a canonical marker

In [ ]:
qz_sorted = sort_latent(adata_norm, 'MYH11')
plot.disp_spatial_latents(adata, qz_sorted, ncols=4)

Next step: Fit a principal curve that sequentially pass through the **sorted** latent (by the Wasserstein distance w.r.t. root marker)?

**TI w/ scFates:**

In [ ]:
%%bash
../env/bin/pip install pandas==1.5.3

In [ ]:
import scFates as scf

In [ ]:
# adata_qz = sc.read_h5ad('../data/xenium/adata_NIH_F5_qz.h5ad')

# adata_norm.obsm['X_z'] = adata_qz.X
# adata_norm.obsm['X_pca'] = adata_qz.obsm['X_pca']

# adata_desi.obsm['X_z'] = adata_qz.X
# adata_desi.obsm['X_pca'] = adata_qz.obsm['X_pca']

In [ ]:
adata_norm.obsm['X_z'] = qz_permute
scf.tl.curve(adata_norm, Nodes=10, use_rep='X_z', ndims_rep=adata_norm.obsm['X_z'].shape[-1])

In [ ]:
scf.pl.graph(adata_norm, basis='z')

In [ ]:
# TI w/ root assignment
scf.tl.root(adata_norm, "MYH11")

In [ ]:
scf.tl.pseudotime(adata_norm, n_jobs=16, n_map=100, seed=0)

In [ ]:
sc.pl.pca(adata_norm, color="t", cmap="coolwarm", title="Pseudotime\n (principal curve)")

In [ ]:
# Spatial distribution of trajectories
sq.pl.spatial_scatter(adata_norm, color='t', size=20, img=False, cmap='coolwarm', title='Spatial Trajectory')

DEGs along trajectory:

In [ ]:
# Xenium
scf.tl.test_association(adata_norm, n_jobs=16)
scf.tl.test_association(adata_norm, reapply_filters=True,A_cut=.5)
scf.pl.test_association(adata_norm)
ti_sig_features = adata_norm.var[adata_norm.var.signi].index

In [ ]:
scf.tl.fit(adata_norm, n_jobs=16)

In [ ]:
scf.pl.single_trend(adata_norm, 'CD34', basis='pca', color_exp='k')

In [ ]:
scf.tl.cluster(adata_norm, n_pcs=4, metric='correlation')
for c in adata_norm.var["clusters"].unique():
    scf.pl.trends(adata_norm, features=adata_norm.var_names[adata_norm.var.clusters==c],basis="pca")

In [ ]:
%%bash
../env/bin/pip install pandas==2.2.2

Held-out validation:

In [ ]:
# adata_val = load_xenium(os.path.join(xenium_path, 'NIH_F4'))
# xenium_coords = adata_val.obs[['y_centroid', 'x_centroid']].copy().to_numpy()

# TODO: need to load paired `u`

G_xenium = gen_graph.construct_graph(xenium_coords, k=30, r=np.inf)
edge_index = pyg_utils.from_networkx(G_xenium).edge_index
qz, qz_Sigma = model.get_z(adata_val.X.A, edge_index)

del G_xenium, edge_index
gc.collect()

In [ ]:
plot.disp_spatial_latents(adata_val, qz, ncols=4)

Baselines: EM clustering:

In [ ]:
from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=10, random_state=0).fit(adata_norm.X.A)
gm_cov = gm.covariances_
gm_soft_assign = gm.predict_proba(adata_norm.X.A)

plot.disp_spatial_latents(adata, gm_soft_assign, ncols=2)

---